# Dog Bounding Box Detection
Uses a pre-trained YOLOv5 model to detect dogs and draw bounding boxes.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2
import numpy as np
from audio_utils import load_data, spectrogram, plot_spectrogram

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO('yolov8n.pt')

# Load font
font = ImageFont.truetype('/Library/Fonts/Arial.ttf', 40)

In [ ]:
# Load dog image
img = Image.open('files/wynnie.jpg').convert('RGB')

# Run inference
results = model(img)

# Draw bounding boxes for dogs
draw = ImageDraw.Draw(img)
dog_count = 0

for result in results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]
        conf = float(box.conf[0])

        if label == 'dog' and conf > 0.5:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            draw.rectangle([x1, y1, x2, y2], outline='blue', width=4)
            draw.rectangle([x1, y1 - 45, x1 + 200, y1], fill='blue')
            draw.text((x1 + 4, y1 - 40), f'{label} {conf:.2f}', fill='white', font=font)

# Display result
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Make dog bark spectrogram
vid = 'files/dogs1.mp4'
s, framerate = load_data(vid)

freqs, times, power, log_spec = spectrogram(vid, s, framerate)
plot_spectrogram(freqs, times, log_spec)

In [ ]:
# Helper to get spectrogram column for time t
def get_spec_col(t):
    return np.argmin(np.abs(times - t))

# Load dog video
cap = cv2.VideoCapture('files/dogs1.mp4')

fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Precompute spectrogram visualization
spec_full = np.flipud(log_spec)

spec_vis_full = cv2.normalize(spec_full, None, 0, 255, cv2.NORM_MINMAX)
spec_vis_full = spec_vis_full.astype(np.uint8)
spec_vis_full = cv2.applyColorMap(spec_vis_full, cv2.COLORMAP_INFERNO)
spec_vis_full = cv2.resize(spec_vis_full, (w, h))

# Video writer for output

out = cv2.VideoWriter(
    'files/dogs1-output.mp4',
    cv2.VideoWriter_fourcc(*'avc1'),
    fps,
    (w * 2, h)
)

frame_id = 0
interval = 3  # Run detection every 3 frames

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t = frame_id / fps # Current time in seconds
    spec_id = get_spec_col(t)

    spec_vis = spec_vis_full.copy()

    x_pos = int(spec_id / log_spec.shape[1] * w)

    cv2.line(spec_vis, (x_pos, 0), (x_pos, h), (0, 255, 255), 3)

    # Run tracking
    if frame_id % interval == 0:
        results = model.track(
            frame,
            persist=True,
            imgsz=640,
            conf=0.4,
            verbose=False
        )

    r = results[0]

    if r.boxes is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        cls_ids = r.boxes.cls.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None

        for i, box in enumerate(boxes):
            label = model.names[int(cls_ids[i])]
            conf = confs[i]

            if label == 'dog' and confs[i] > 0.5:
                x1, y1, x2, y2 = map(int, box)

                track_id = int(ids[i]) if ids is not None else -1

                # Draw box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)

                text = f'{label} #{track_id} {conf:.2f}'

                cv2.rectangle(frame, (x1, y1 - 45), (x1 + 100, y1), (255, 0, 0), -1)
                cv2.putText(frame, text,
                            (x1 + 5, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255, 255, 255), 2)

    frame = frame.astype('uint8')
    frame = np.ascontiguousarray(frame)

    frame = cv2.resize(frame, (w, h))
    spec_vis = cv2.resize(spec_vis, (w, h))
    frame = np.hstack([frame, spec_vis])

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
cv2.destroyAllWindows()